In [1]:
import requests


username = "torvalds"

In [2]:
# Construir la URL del endpoint de usuario
user_url = f"https://api.github.com/users/{username}"

# Hacer la solicitud GET a la API de GitHub para obtener datos del perfil
response = requests.get(user_url)

# Verificar si la solicitud fue exitosa (código 200)
if response.status_code != 200:
    print(f"Error al obtener datos del usuario: {response.status_code}")
else:
    user_data = response.json()  # Convertir la respuesta JSON a un diccionario de Python

    # Extraer campos importantes del perfil
    name = user_data.get("name")
    login = user_data.get("login")        # nombre de usuario (debería ser igual a username)
    bio = user_data.get("bio")
    public_repos = user_data.get("public_repos")
    followers = user_data.get("followers")
    following = user_data.get("following")
    location = user_data.get("location")
    profile_url = user_data.get("html_url")

    # Imprimir datos de perfil (formato básico, refinaremos formato más adelante)
    print("GitHub Profile:", login)
    print("Name:", name)
    print("Username:", login)
    print("Bio:", bio)
    print("Public Repos:", public_repos)
    print("Followers:", followers)
    print("Following:", following)
    print("Location:", location)
    print("Profile URL:", profile_url)


GitHub Profile: torvalds
Name: Linus Torvalds
Username: torvalds
Bio: None
Public Repos: 8
Followers: 243336
Following: 0
Location: Portland, OR
Profile URL: https://github.com/torvalds


In [3]:
# Construir la URL para obtener repositorios, solicitando hasta 100 repos por página
repos_url = f"https://api.github.com/users/{username}/repos?per_page=100"

# Hacer la solicitud GET para obtener la lista de repositorios
response_repos = requests.get(repos_url)

if response_repos.status_code != 200:
    print(f"Error al obtener repositorios: {response_repos.status_code}")
else:
    repos_data = response_repos.json()  # lista de diccionarios, cada uno es un repo

    # Lista para almacenar info resumida de cada repositorio
    repos_info_list = []

    for repo in repos_data:
        repo_name = repo.get("name")
        repo_url = repo.get("html_url")
        stars = repo.get("stargazers_count")
        language = repo.get("language")
        last_update = repo.get("updated_at")

        # Vamos a formatear la fecha de última actualización para mostrar solo la fecha (YYYY-MM-DD)
        if last_update:
            last_update = last_update.split("T")[0]  # toma solo la parte antes de 'T'

        # Guardar la info en un diccionario
        repo_info = {
            "name": repo_name,
            "url": repo_url,
            "stars": stars,
            "language": language,
            "last_update": last_update
        }
        repos_info_list.append(repo_info)

    # Imprimir para verificar (formato básico)
    print("Repositories ---------------------")
    for info in repos_info_list:
        print(info["name"])
        print(" View Repo:", info["url"])
        print(" Stars:", info["stars"])
        print(" Language:", info["language"])
        print(" Last Updated:", info["last_update"])
        print()  # línea en blanco entre repos


Repositories ---------------------
1590A
 View Repo: https://github.com/torvalds/1590A
 Stars: 389
 Language: OpenSCAD
 Last Updated: 2025-08-11

libdc-for-dirk
 View Repo: https://github.com/torvalds/libdc-for-dirk
 Stars: 274
 Language: C
 Last Updated: 2025-08-03

libgit2
 View Repo: https://github.com/torvalds/libgit2
 Stars: 209
 Language: C
 Last Updated: 2025-08-04

linux
 View Repo: https://github.com/torvalds/linux
 Stars: 199404
 Language: C
 Last Updated: 2025-08-11

pesconvert
 View Repo: https://github.com/torvalds/pesconvert
 Stars: 443
 Language: C
 Last Updated: 2025-07-27

subsurface-for-dirk
 View Repo: https://github.com/torvalds/subsurface-for-dirk
 Stars: 337
 Language: C++
 Last Updated: 2025-07-27

test-tlb
 View Repo: https://github.com/torvalds/test-tlb
 Stars: 806
 Language: C
 Last Updated: 2025-08-05

uemacs
 View Repo: https://github.com/torvalds/uemacs
 Stars: 1524
 Language: C
 Last Updated: 2025-08-11



In [4]:
# Diccionario para acumular bytes de código por lenguaje en todos los repos
total_bytes_by_lang = {}

for repo in repos_info_list:
    repo_name = repo["name"]
    # Construir la URL de lenguajes para este repositorio
    lang_url = f"https://api.github.com/repos/{username}/{repo_name}/languages"

    # Hacer la petición a la URL de lenguajes
    response_langs = requests.get(lang_url)
    if response_langs.status_code != 200:
        print(f"Error obteniendo lenguajes de {repo_name}: {response_langs.status_code}")
        continue  # saltar al siguiente repo si hay error
    languages_data = response_langs.json()  # diccionario de lenguajes y bytes

    # Sumar los bytes de cada lenguaje al total global
    for lang, bytes_count in languages_data.items():
        if lang in total_bytes_by_lang:
            total_bytes_by_lang[lang] += bytes_count
        else:
            total_bytes_by_lang[lang] = bytes_count

# Calcular porcentajes de uso de cada lenguaje
most_used_languages = []
if total_bytes_by_lang:
    total_bytes = sum(total_bytes_by_lang.values())
    for lang, bytes_count in total_bytes_by_lang.items():
        percentage = (bytes_count / total_bytes) * 100
        # Guardar tupla (porcentaje, lenguaje) para ordenar después
        most_used_languages.append((percentage, lang))

    # Ordenar de mayor a menor porcentaje
    most_used_languages.sort(reverse=True, key=lambda x: x[0])

    # Preparar lista de strings "Lenguaje (X%)"
    lang_output_list = []
    for perc, lang in most_used_languages:
        # Formatear el porcentaje con 1 decimal
        perc_str = f"{perc:.1f}%"
        # Opcional: podemos redondear .0 para mostrarlo como entero si no tiene decimal
        if perc_str.endswith(".0%"):
            perc_str = perc_str.replace(".0%", "%")
        lang_output_list.append(f"{lang} ({perc_str})")

    # Generar el string final de lenguajes más usados
    languages_output = ", ".join(lang_output_list)

    # Imprimir la línea de "Most Used Languages"
    print("Most Used Languages")
    print(languages_output)


Most Used Languages
C (97.9%), Assembly (0.7%), Shell (0.4%), Python (0.3%), C++ (0.2%), Makefile (0.2%), Rust (0.1%), Perl (0.1%), XSLT (0%), QML (0%), Roff (0%), SmPL (0%), CMake (0%), Yacc (0%), Awk (0%), Lex (0%), JavaScript (0%), HTML (0%), Jinja (0%), M4 (0%), Java (0%), TeX (0%), UnrealScript (0%), CSS (0%), QMake (0%), Gherkin (0%), OpenSCAD (0%), Linker Script (0%), Dockerfile (0%), Objective-C++ (0%), R (0%), Clojure (0%), MATLAB (0%), sed (0%), XS (0%), RPC (0%), Batchfile (0%)


In [5]:
def format_star_count(stars):
    """Convierte un número de estrellas en formato abreviado con K/M si es grande."""
    if stars is None:
        return "0"
    if stars < 1000:
        return str(stars)
    elif stars < 1000000:
        # Formato K (miles)
        # Dividir por 1000.0 para obtener decimal, una cifra decimal si es necesario
        k = stars / 1000.0
        # Si el valor tiene decimal .0, mostrar entero, si no, una cifra decimal
        if k.is_integer():
            return f"{int(k)}K"
        else:
            return f"{k:.1f}K"
    else:
        # Formato M (millones) por si algún repositorio tuviera >= 1 millón de estrellas
        m = stars / 1000000.0
        if m.is_integer():
            return f"{int(m)}M"
        else:
            return f"{m:.1f}M"

# Ordenar los repositorios por número de estrellas descendente
repos_sorted_by_stars = sorted(repos_info_list, key=lambda x: x["stars"] or 0, reverse=True)

# Tomar top 3 repos (o menos si no hay tantos)
top_repos = repos_sorted_by_stars[:3]

# Imprimir encabezado de Most Starred Repos
print("Most Starred Repos")
for repo in top_repos:
    name = repo["name"]
    star_count = repo["stars"] or 0
    star_str = format_star_count(star_count)
    print(f"{name} – {star_str} stars")


Most Starred Repos
linux – 199.4K stars
uemacs – 1.5K stars
test-tlb – 806 stars


In [6]:
import requests

username = "torvalds"

# 1. Obtener datos de perfil del usuario
user_url = f"https://api.github.com/users/{username}"
response = requests.get(user_url)
if response.status_code != 200:
    print(f"Error al obtener datos del usuario: {response.status_code}")
else:
    user_data = response.json()
    name = user_data.get("name")
    login = user_data.get("login")
    bio = user_data.get("bio")
    public_repos = user_data.get("public_repos")
    followers = user_data.get("followers")
    following = user_data.get("following")
    location = user_data.get("location")
    profile_url = user_data.get("html_url")

    # 2. Obtener lista de repositorios del usuario
    repos_url = f"https://api.github.com/users/{username}/repos?per_page=100"
    response_repos = requests.get(repos_url)
    if response_repos.status_code != 200:
        print(f"Error al obtener repositorios: {response_repos.status_code}")
    else:
        repos_data = response_repos.json()
        repos_info_list = []
        for repo in repos_data:
            repo_name = repo.get("name")
            repo_url = repo.get("html_url")
            stars = repo.get("stargazers_count")
            language = repo.get("language")
            last_update = repo.get("updated_at")
            if last_update:
                last_update = last_update.split("T")[0]
            repos_info_list.append({
                "name": repo_name,
                "url": repo_url,
                "stars": stars,
                "language": language,
                "last_update": last_update
            })

        # 3. Calcular lenguajes más usados
        total_bytes_by_lang = {}
        for repo in repos_info_list:
            repo_name = repo["name"]
            lang_url = f"https://api.github.com/repos/{username}/{repo_name}/languages"
            resp_lang = requests.get(lang_url)
            if resp_lang.status_code == 200:
                for lang, bytes_count in resp_lang.json().items():
                    total_bytes_by_lang[lang] = total_bytes_by_lang.get(lang, 0) + bytes_count
        most_used_langs_str = ""
        if total_bytes_by_lang:
            total_bytes = sum(total_bytes_by_lang.values())
            # Obtener porcentaje y ordenar
            lang_percentages = []
            for lang, bytes_count in total_bytes_by_lang.items():
                perc = (bytes_count / total_bytes) * 100
                lang_percentages.append((perc, lang))
            lang_percentages.sort(reverse=True, key=lambda x: x[0])
            # Formatear los dos primeros lenguajes (por ejemplo) o todos si se prefiere
            formatted_langs = []
            for perc, lang in lang_percentages[:len(lang_percentages)]:
                perc_str = f"{perc:.1f}%"
                if perc_str.endswith(".0%"):
                    perc_str = perc_str.replace(".0%", "%")
                formatted_langs.append(f"{lang} ({perc_str})")
            most_used_langs_str = ", ".join(formatted_langs)

        # 4. Calcular repos más estrellados (top 3)
        repos_sorted_by_stars = sorted(repos_info_list, key=lambda x: x["stars"] or 0, reverse=True)
        top_repos = repos_sorted_by_stars[:3]
        def format_star_count(stars):
            if stars is None:
                return "0"
            if stars < 1000:
                return str(stars)
            elif stars < 1000000:
                k = stars / 1000.0
                return (f"{k:.1f}K" if not k.is_integer() else f"{int(k)}K")
            else:
                m = stars / 1000000.0
                return (f"{m:.1f}M" if not m.is_integer() else f"{int(m)}M")

        # 5. Imprimir todos los resultados formateados
        print(f"# GitHub Profile: {login}")
        print(f"# Name: {name}")
        print(f"# Username: {login}")
        print(f"# Bio: {bio if bio else ''}")
        print(f"# Public Repos: {public_repos}")
        print(f"# Followers: {followers:,}".replace(",", "K") if followers and followers >= 1000 else f"# Followers: {followers}")
        print(f"# Following: {following}")
        print(f"# Location: {location if location else ''}")
        print(f"# Profile URL: {profile_url}")
        print("\n# Repositories ---------------------\n")
        for repo in repos_info_list:
            print(f"# {repo['name']}")
            print(f"#  View Repo ({repo['url']})")
            star_str = format_star_count(repo['stars'])
            print(f"#  Stars: {star_str}")
            print(f"#  Language: {repo['language'] if repo['language'] else 'N/A'}")
            print(f"#  Last Updated: {repo['last_update']}")
            print("")  # línea en blanco después de cada repositorio
        print("\n# Most Used Languages")
        print(f"# {most_used_langs_str}")
        print("\n# Most Starred Repos")
        for repo in top_repos:
            star_str = format_star_count(repo['stars'])
            print(f"# {repo['name']} – {star_str} stars")


# GitHub Profile: torvalds
# Name: Linus Torvalds
# Username: torvalds
# Bio: 
# Public Repos: 8
# Followers: 243K336
# Following: 0
# Location: Portland, OR
# Profile URL: https://github.com/torvalds

# Repositories ---------------------

# 1590A
#  View Repo (https://github.com/torvalds/1590A)
#  Stars: 389
#  Language: OpenSCAD
#  Last Updated: 2025-08-11

# libdc-for-dirk
#  View Repo (https://github.com/torvalds/libdc-for-dirk)
#  Stars: 274
#  Language: C
#  Last Updated: 2025-08-03

# libgit2
#  View Repo (https://github.com/torvalds/libgit2)
#  Stars: 209
#  Language: C
#  Last Updated: 2025-08-04

# linux
#  View Repo (https://github.com/torvalds/linux)
#  Stars: 199.4K
#  Language: C
#  Last Updated: 2025-08-11

# pesconvert
#  View Repo (https://github.com/torvalds/pesconvert)
#  Stars: 443
#  Language: C
#  Last Updated: 2025-07-27

# subsurface-for-dirk
#  View Repo (https://github.com/torvalds/subsurface-for-dirk)
#  Stars: 337
#  Language: C++
#  Last Updated: 2025-07-27